# 2 – Análisis de ejes biológicos (refinado para la firma del grafo)

Dos niveles de análisis:
1. **Cobertura funcional** de la firma del grafo (estructural)
2. **Scores por muestra** sobre la expresión completa (funcional por subtipo)

In [ ]:
import os, sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import matplotlib.pyplot as plt

from intergate.config import CFG
from intergate.utils import set_seed, set_all_seeds, cleanup_memory
from intergate.data import (
    load_expression_and_metadata, prepare_genes, encode_labels,
    cohort_split, scale_features, apply_connected_only,
    make_dataloaders, build_regulator_features, ExpressionDataset,
)
from intergate.graph import build_backbone
from intergate.graph_cache import get_or_build_backbone, get_or_build_Xh
from intergate.losses import compute_metrics_full, compute_class_weights_balanced
from intergate.training import predict_proba_xh_mode
from intergate.pruning import export_pruned_graph
from intergate.ablation import (
    AblationConfig, load_bundle, build_pruned_model_from_bundle,
    register_runtime, get_full_gene_matrix_and_genes,
    run_single_seed, run_ablation,
)

set_seed(CFG.SEED)
DEVICE = CFG.DEVICE
print("DEVICE:", DEVICE)


In [ ]:
from intergate.axes import (
    compute_axis_scores, plot_axis_boxplots,
    plot_axis_heatmap_by_subtype, plot_axis_correlation,
    plot_axis_embedding, axis_stats_by_subtype,
    one_vs_rest_axis_analysis, summarize_one_vs_rest,
    plot_one_vs_rest_heatmap, summarize_axis_overlap_with_graph,
)


## 1) Data pipeline

In [ ]:
X_df, y_str, cohort = load_expression_and_metadata(
    CFG.EXPR_CSV, CFG.META_CSV,
    sample_col=CFG.SAMPLE_COL, label_col=CFG.LABEL_COL, cohort_col=CFG.COHORT_COL,
)
X_df_kegg, genes_kegg = prepare_genes(X_df)
y, classes, label_map = encode_labels(y_str)
n_classes = len(classes)


In [ ]:
edge_index, edge_weight, edge_type = get_or_build_backbone(
    genes_kegg, cache_dir=CFG.PIPELINE_CACHE_DIR,
    force_rebuild=False, use_omnipath=CFG.USE_OMNIPATH, use_huri=CFG.USE_HURI,
)
print(f"Backbone: {edge_index.shape[1]} aristas, {len(genes_kegg)} genes")


In [ ]:
train_idx, val_idx, test_idx = cohort_split(
    cohort, y, train_cohort_frac=0.80, val_size=CFG.VAL_SIZE, seed=CFG.SEED,
)
Xs_gene = scale_features(X_df_kegg, train_idx, mode=CFG.SCALE_MODE, use_quantile=CFG.USE_QUANTILE)

if CFG.CONNECTED_ONLY:
    Xs_gene, edge_index, edge_weight, edge_type, genes_kegg = apply_connected_only(
        Xs_gene, edge_index, edge_weight, edge_type, genes_kegg,
    )


In [ ]:
Xs_graph, graph_feat_names = get_or_build_Xh(
    Xs_gene, genes_kegg, edge_index,
    cache_dir=CFG.PIPELINE_CACHE_DIR,
    stats=CFG.REG_STATS, min_targets=CFG.REG_MIN_GENES,
    max_regulators=CFG.REG_MAX_REGULATORS,
)


In [ ]:
dl_tr, dl_va, dl_te = make_dataloaders(
    Xs_gene, Xs_graph, y, train_idx, val_idx, test_idx, batch_size=CFG.BATCH_SIZE,
)


In [ ]:
edge_index_t = torch.as_tensor(edge_index, dtype=torch.long, device=DEVICE)
edge_weight_t = torch.as_tensor(edge_weight, dtype=torch.float32, device=DEVICE)
edge_type_t = torch.as_tensor(edge_type, dtype=torch.long, device=DEVICE)

register_runtime(
    Xs_gene=Xs_gene, genes_kegg=genes_kegg, X_h=Xs_graph,
    y=y, n_classes=n_classes, label_map=label_map,
    train_idx=train_idx, val_idx=val_idx, test_idx=test_idx,
    edge_index_t=edge_index_t, edge_weight_t=edge_weight_t, edge_type_t=edge_type_t,
)


## 2) Definir ejes biológicos

In [ ]:
AXES = {
    "Luminal Hormonal": [
        "ESR1","PGR","GATA3","GREB1","ESR2","PHLDA1","LRIG1","RXRA"
    ],
    "Cell Cycle Mitotic": [
        "AURKA","AURKB","BUB1","CDC7","CDC23","CDCA3","CDK1","CDK2",
        "CEP55","E2F1","FOXM1","PLK1","TTK","ZWINT"
    ],
    "HER2 RTK MAPK": [
        "ERBB2","EGFR","IGF1R","MET","PDGFRB","AKT1","MAP2K1","MAP2K2",
        "MAPK1","MAPK3","MAPK14","PLCG1","SRC","LYN"
    ],
    "Basal Plasticity TNBC": [
        "KRT6A","KRT16","SOX10","VGLL1","VGLL3","EGFR","EMP1","MSLN"
    ],
    "Immune Lymphoid Signaling": [
        "CD79A","CXCL9","DEF6","GRAP2","HCK","JAK3","ITPKB","PAX5",
        "SYK","S1PR1","SOCS1","TRAF1","TRAF2","ZBP1"
    ],
    "DNA Damage p53 Checkpoint": [
        "ATM","ERCC3","FANCG","KAT5","PRKDC","TP53","MDM2","PTEN",
        "DAPK1","STK11","SIRT1"
    ],
    "Adhesion Cytoskeleton Invasion": [
        "ABI2","FLNA","ITGB1","ITGB1BP1","MMP2","PTK2","RHOA","ROCK1",
        "LASP1","NCK1","NCK2","SDCBP"
    ],
    "Androgen Apocrine": [
        "AR","MSLN"
    ],
}
AXES_ORDER = list(AXES.keys())


## 3) Cobertura de ejes en la firma del grafo

In [ ]:
# Genes del grafo conectado
graph_genes = set(g.upper() for g in genes_kegg)
overlap_df = summarize_axis_overlap_with_graph(graph_genes, AXES)
display(overlap_df)


## 4) Scores por muestra (expresión completa)

In [ ]:
df_expr_raw = pd.read_csv(CFG.EXPR_CSV, index_col=0)
metadata_axes = pd.read_csv(CFG.META_CSV, index_col=0)[["sample", "batch", "label"]]

meta_scores, axis_info, Xz = compute_axis_scores(
    expr_df=df_expr_raw, meta_df=metadata_axes,
    axes=AXES, sample_col="sample", label_col="label",
)
display(axis_info[["Axis", "n_present", "coverage", "present_genes"]])


## 5) Figuras

### 5.1) Figure 2

In [ ]:
SAVE = "./figures_axes_refined"

In [ ]:
# ── A) Heatmap de medias por subtipo ──
heat = plot_axis_heatmap_by_subtype(
    meta_scores, dpi=300, label_col="label",vmin=-1, vmax=1,figure_size=(14, 10),xtick_align="end",#start,center,end
    title="",            title_fontsize=16,   bold_title=False,color_brightness=0.9,
    xtick_fontsize=24,   bold_xticks=False,
    ytick_fontsize=28,   bold_yticks=False,
    annot_fontsize=23,   bold_annot=False,
    colorbar_label_fontsize=28, colorbar_tick_fontsize=23, 
    bold_colorbar_label=False,
    colorbar_label_rotation=90,
    colorbar_label_pad=0,
    y_order=["Normal", "LumA", "LumB", "HER2", "TNBC"],
    panel_label="A", panel_label_fontsize=42, panel_label_bold=False,
    panel_label_x=0.016, panel_label_y=1.04,
    save_dir=SAVE,
    save_name="fig_axis_heatmap_means_by_subtype", 
    save_format="png"
)



In [ ]:
# ── B) Correlación Spearman ──
corr = plot_axis_correlation(
    meta_scores, dpi=300,figure_size=(14, 10),xtick_align="end",#start,center,end
    title="",            title_fontsize=16,   bold_title=False,color_brightness=0.9,
    xtick_fontsize=24,   bold_xticks=False,
    ytick_fontsize=24,   bold_yticks=False,
    annot_fontsize=23,   bold_annot=False,
    colorbar_label_fontsize=26, colorbar_tick_fontsize=22, 
    bold_colorbar_label=False,
    colorbar_label_rotation=90,
    colorbar_label_pad=0,
    panel_label="B", panel_label_fontsize=42, panel_label_bold=False,
    panel_label_x=0.01, panel_label_y=1.04,
    save_dir=SAVE,
    save_name="fig_axis_corr_spearman", 
    save_format="png"
)





In [ ]:
# ── C/D) PCA + UMAP ──
plot_axis_embedding(
    meta_scores, dpi=300, label_col="label",figure_size=(12, 8),color_brightness=0.9,
    pca_title="",        umap_title="",       title_fontsize=16,   bold_title=False,
    xlabel_fontsize=28,  bold_xlabel=False,
    ylabel_fontsize=28,  bold_ylabel=False,
    xtick_fontsize=24,   bold_xticks=False,
    ytick_fontsize=24,   bold_yticks=False,
    legend_fontsize=16,  bold_legend=False,
    legend_markersize=15,           # tamaño círculos leyenda (pts)
    signed_xticks=True,            # +/− en eje X
    signed_yticks=True,            # +/− en eje Y
    xtick_decimals=0,
    ytick_decimals=0,
    scatter_size=21,
    pca_panel_label="C", umap_panel_label="D",
    panel_label_fontsize=42, panel_label_bold=False,
    panel_label_x=0.016, panel_label_y=1.04,
    save_dir=SAVE,
    save_format="png",
    pca_save_name="fig_axis_pca_space",
    umap_save_name="fig_axis_umap_space",
)



In [ ]:
# ── E) One-vs-rest heatmap ──
ovr_df = one_vs_rest_axis_analysis(meta_scores, label_col="label", axes_order=AXES_ORDER)

plot_one_vs_rest_heatmap(
    ovr_df, dpi=300,vmin=-1, vmax=1,figure_size=(12, 8),xtick_align="end",#start,center,end
    value_col="delta_mean", fdr_col="p_fdr_subtype",color_brightness=0.9,
    title="",            title_fontsize=16,   bold_title=False,
    xtick_fontsize=24,   bold_xticks=False,
    ytick_fontsize=28,   bold_yticks=False,
    annot_fontsize=23,   bold_annot=False,
    colorbar_label_fontsize=28, colorbar_tick_fontsize=23, 
    bold_colorbar_label=False,
    colorbar_label_rotation=90,
    colorbar_label_pad=0,
    y_order=["Normal", "LumA", "LumB", "HER2", "TNBC"],
    panel_label="D", panel_label_fontsize=42, panel_label_bold=False,
    panel_label_x=0.01, panel_label_y=1.04,
    save_dir=SAVE,
    save_name="fig_axis_ovr_heatmap_delta_mean",
    save_format="png"
)

### 5.2) Figure Suplementary

In [ ]:
plot_axis_boxplots(meta_scores, label_col="label")
stats_df = axis_stats_by_subtype(meta_scores, label_col="label")
display(stats_df)


## 6) One-vs-rest por subtipo

In [ ]:
ovr_df = one_vs_rest_axis_analysis(meta_scores, label_col="label", axes_order=AXES_ORDER)
display(ovr_df.head(20))

ovr_summary = summarize_one_vs_rest(ovr_df, top_n=3, sort_by="delta_mean")
display(ovr_summary)

plot_one_vs_rest_heatmap(ovr_df, title="", dpi=300,value_col="delta_mean", fdr_col="p_fdr_subtype", tick_fontsize=12, annot_fontsize=12,save_dir=SAVE)


# Fin